# ExaMLOps — Platform Ops Starter

This is the **governed platform-management workbench**. From here you manage the platform *itself* —
the compute-node cost, connections, the ExaMLOps↔bridge wiring, service config, and `platform.db`
knobs — and you deploy calculation code to the platform.

**Everything goes through one façade, `examlops.platform_admin`.** It is not a convenience wrapper: it
is what *governs* your change. Every call runs **RBAC → policy → audit**, so a change you make here is
attributable to you, policy-gated, and on the tamper-evident audit chain — exactly like a change made
with the `exa` CLI or the dashboard. Writing to the config files / provider store / DB *directly* would
bypass that — so always go through `pa.*`.

| Tier | What | How it ships |
|---|---|---|
| **A** (default) | config (cost, connections, bridge UUID, knobs) + **sandboxed** calc code (providers) | hot, no restart, reversible |
| **B** (admin) | real integration **source** (e.g. the bridge) | staged → `dualgit ship` → service redeploy |

## 1. Who am I, and what tier am I?

In [ ]:
import os
from examlops import platform_admin as pa

print('actor        :', os.environ.get('EXAMLOPS_ACTOR', '(unset)'))
print('audit source :', os.environ.get('EXAMLOPS_ADMIN_SOURCE', 'workbench'))
print('config dir   :', os.environ.get('EXAMLOPS_CONFIG_DIR', '~/.config/examlops'))
print('Tier B (source editing) available:', os.environ.get('EXAMLOPS_PLATFORM_SOURCE') == '1')

## 2. Tier A — change the compute-node cost

The example the design was built around. This writes the rate card the platform's cost estimator reads,
so `exa models cost` and the dashboard FinOps view pick it up immediately — and it's audited.

In [ ]:
print('before:', pa.compute_cost_card())

# --- apply (uncomment) ---
# pa.set_compute_cost(gpu_per_hour=3.10, cpu_per_hour=0.07)
# print('after :', pa.compute_cost_card())

## 3. Tier A — write Python and deploy it to the platform

A **provider** is a swappable calculation (cost / carbon / drift / promotion …). You write the Python
here; `deploy_provider` runs it through the **AST security gate** *before* it reaches disk (imports,
dunders and dangerous calls are rejected), persists it, and activates it. The next cost calculation
uses it — no restart. Provenance shows up in `exa providers list` and the dashboard Providers console.

In [ ]:
code = '''
class WeekendDiscountCost(Provider):
    name = "weekend-discount"
    def compute(self, inputs):
        gpu_h = inputs.get("gpu_hours", 0)
        rate = inputs.get("gpu_rate", 2.5)
        return {"cost_usd": round(gpu_h * rate * 0.8, 4)}  # 20% off
'''

# --- deploy (uncomment) ---
# info = pa.deploy_provider('cost', 'weekend-discount', code)
# print(info['result'])
# print(pa.list_authored_providers())

## 4. Tier A — other config surfaces

All governed + audited, all reusing the exact code path the `exa` CLI uses.

In [ ]:
# Connections (ExaMLOps ↔ data-plane / S3 / URI) — secret goes to the secrets store, never the DB/audit:
# pa.set_connection('dataplane-prod', 'dataplane', config={'endpoint': 'http://dataplane:8000'}, secret_value='...')

# Bridge ↔ model UUID mapping:
# pa.set_bridge_uuid('JPCP')                 # assign if missing (idempotent)
# pa.set_bridge_uuid('JPCP', regenerate=True)  # rotate (HPC teams must update their config)

# platform.db knobs (traffic split, autoscale, …):
# pa.set_knob('traffic', 'JPCP', {'Production': 90, 'Canary': 10})

# Service URLs / tokens (token values are redacted in the audit trail):
# pa.set_config(control_plane_url='http://control-plane:8002')

## 5. Tier B — edit the ExaMLOps↔bridge / data-plane source (admin)

Editing real integration code is deliberately **not** hot-applied. Edit the file under
`/repo/platform/clients/` (rw only in an admin platform-ops workbench), then record the intent — which
returns the exact ship + redeploy steps. Nothing is committed or restarted for you.

In [ ]:
# after editing /repo/platform/clients/seanerbus_bridge.py in the file browser:
# out = pa.propose_source_change(['platform/clients/seanerbus_bridge.py'], 'tune bridge retry/backoff')
# for step in out['result']['next_steps']:
#     print('-', step)
#
# Then ask the assistant: "commit and redeploy my platform-ops source change".

## 6. See the results — the change feed

Every change you made above is on the audit chain and shows in the dashboard **Platform Ops** console.
Here's the same feed from Python:

In [ ]:
for row in pa.recent_changes(limit=15):
    print(f"{row['ts']}  {row['actor']:10}  {row['action']:35}  {row['target']}")